# GraphRAG vs Baseline Text-to-SQL — Evaluation Pipeline (Kaggle)

Notebook ini menjalankan **dua pipeline Text-to-SQL** pada Spider dev set, lalu mengevaluasinya dengan lapisan metrik skripsi. Semua logika ada di `src/` — notebook ini memanggilnya langkah demi langkah dan **menjelaskan apa yang terjadi di tiap langkah**, jadi bisa dibaca sebagai dokumentasi cara kerja pipeline.

## Dua pipeline yang dibandingkan

| Tahap | GraphRAG (`pipeline.py` → `run_single`) | Baseline (`baseline.py` → `run_single_baseline`) |
|---|---|---|
| Input | pertanyaan | pertanyaan |
| Pilih tabel | Stage 1: **tabel** dengan skor ≥ threshold tabel (n-gram + cosine BGE-M3) **sama** dengan Stage 1 GraphRAG (n-gram + cosine, skor ≥ threshold tabel), tapi dicocokkan ke teks tabel `table <nama> columns <kolom...>` |
| Pilih kolom | Stage 2: **kolom** di tabel kandidat dengan skor ≥ threshold kolom, + kolom dari **value linking** (nilai sel yang disebut di pertanyaan) | — (tidak ada) |
| Jalur JOIN | Graph traversal: shortest path antar **kolom**; yang masuk prompt hanya **kolom JOIN** (`fk_only`) | Table path tracing: shortest path antar **tabel** via FK |
| Schema context | DDL hanya **kolom terpilih** | DDL **semua kolom** tabel terpilih (tanpa pruning) |
| Few-shot | top-k contoh dari `train_spider.json`, dipilih by **skeleton** pertanyaan | **sama persis** (`cfg.few_shot_k`, index yang sama) |
| Prompt → LLM → cleaner | Prompt builder → Qwen2.5-Coder (4-bit, greedy) → SQL cleaner | **sama persis** untuk keduanya |

Kedua pipeline memakai LLM dan few-shot yang **sama**; yang berbeda hanya retrieval-nya.

GraphRAG memakai **preprocessing tetap** hasil trial SLA (`notebooks/graphrag_sla_trial.ipynb`, trial G7); yang bisa dipilih hanya threshold tabel dan kolom (sel 3). Few-shot dipilih dengan **skeleton** pertanyaan (`src/skeleton/`, `few_shot_selection="skeleton"`).

## Tiga tahap ablation (keputusan `IMPLEMENTATION_DECISIONS.md` poin 28)

| Tahap | Yang dicari | Grid (`config.py`) | Dipilih dengan | Jalankan |
|---|---|---|---|---|
| 1 | threshold tabel & kolom (dipakai kedua pipeline untuk tabel; kolom hanya GraphRAG) | grid di `graphrag_sla_trial.ipynb` | **SLA** (tanpa LLM) | `graphrag_sla_trial.ipynb` |
| 2 | few-shot k (sama untuk kedua pipeline) | `few_shot_frame` | **EX** | `ablation.py --mode {graphrag,baseline}` |
| 3 | model LLM | `llm_model_frame` | EX (+ token) | `MODEL` di sel 3, atau `pipeline.py --baseline --models all` |

Notebook ini menjalankan **satu** konfigurasi (satu model) dari ujung ke ujung; tahap 2 dan 3 lebih praktis lewat CLI karena butuh banyak run LLM.

## Alur notebook ini

| Bagian | Isi |
|---|---|
| 1-3 | Setup Kaggle, import, konfigurasi (`PipelineConfig` = sumber kebenaran tunggal) |
| 4-7 | Load data, bangun graph, load model, bangun index |
| 8 | Setup evaluasi (parser Spider, foreign-key map, gold schema) |
| 9-10 | **Walk-through 1 sampel**: tiap tahap pipeline diprint, biar kelihatan cara kerjanya |
| 11 | **Loop utama**: jalankan kedua pipeline atas semua sampel + evaluasi per query |
| 12 | Simpan prediksi dan `raw_logs` |
| 13 | **Hasil 6 metrik**: SLA, ESM, EX, CM, Token + TEP, QVT (`metrics_results.json`) |
| 14 | Cross-check evaluator resmi Spider |
| 15 | **Analisis 6 dimensi**, lalu **simpan semua hasil** ke JSON, Excel, dan zip |

## Cara memakai untuk tiga model

Tiap orang hanya mengganti `MODEL` di sel Bagian 3 (`"3B"`, `"7B"`, atau `"14B"`), lalu menjalankan semua sel. Notebook selalu menjalankan **1-Stage** (`baseline` di kode) dan **2-Stage** (`graphrag` di kode). Hasil tiap model masuk ke folder sendiri, lalu digabung dengan `combine_models.py` (Bagian 15e).

Semua hasil ditampilkan sebagai tabel angka (`show()` dari `src/experiments/result_tables.py`, modul yang sama dengan export Excel). Notebook ini **tidak menulis interpretasi atau kesimpulan**; pembacaan hasil dilakukan manual.

> **Catatan run final:** untuk skripsi, set `SAMPLE_RATIO = 1.0` (full 1034 query). Data QVT (`build_qvt_variations.py`) menolak `raw_logs` hasil subsampling, karena `query_id` = posisi baris di `dev.json`.

In [ ]:
# ============================================================
# 1. SETUP — clone repo, install dependency, cek evaluator Spider
# Sel ini aman dijalankan ulang (semua langkah dicek dulu).
# ============================================================
# os untuk pindah folder, subprocess untuk menjalankan perintah terminal
import os, subprocess
# Path untuk menulis lokasi file dengan aman di semua sistem operasi
from pathlib import Path

# Alamat repo yang di-clone saat notebook jalan di Kaggle
REPO_URL = "https://github.com/william-darma-wijaya/thesis-2803.git"
# Branch repo yang dipakai
REPO_BRANCH = "main"

# Fungsi pembantu untuk menjalankan satu perintah terminal
def sh(cmd: str) -> None:
    # Tampilkan perintahnya supaya terlihat apa yang sedang dijalankan
    print(f"$ {cmd}")
    # Jalankan perintah; check=True membuat notebook berhenti kalau perintahnya gagal
    subprocess.run(cmd, shell=True, check=True)

# Fungsi pembantu untuk mengecek apakah sebuah folder adalah root repo ini
def is_repo_root(p: Path) -> bool:
    # Cek file milik repo ini, BUKAN cuma folder "src": Kaggle punya /kaggle/src sendiri,
    # jadi cek "../src" dari /kaggle/working salah mengira /kaggle sebagai root repo.
    return (p / "src" / "core" / "config.py").is_file()

# 1) cwd HARUS root repo: pipeline memakai path relatif (outputs/, data/raw_logs/, external/spider_eval/)
# Kalau folder sekarang bukan root repo, cari atau siapkan dulu
if not is_repo_root(Path.cwd()):
    # Kasus pertama: folder induknya adalah root repo
    if is_repo_root(Path.cwd().parent):  # notebook dijalankan dari notebooks/
        # Naik satu folder ke root repo
        os.chdir("..")
    # Kasus kedua: repo berada di subfolder atau belum ada
    else:                                # Kaggle: repo belum ada di /kaggle/working
        # Repo belum pernah di-clone
        if not is_repo_root(Path("thesis-2803").resolve()):
            # Clone repo dari GitHub
            sh(f"git clone -b {REPO_BRANCH} {REPO_URL}")
        # Repo sudah ada dari run sebelumnya
        else:                                # sudah pernah di-clone: pastikan di branch yang benar
            # Ambil versi terbaru dari branch yang dipilih
            sh(f"git -C thesis-2803 fetch origin {REPO_BRANCH} && git -C thesis-2803 checkout {REPO_BRANCH} && git -C thesis-2803 pull origin {REPO_BRANCH}")
        # Masuk ke folder repo
        os.chdir("thesis-2803")
# Simpan lokasi root repo untuk dipakai sel berikutnya
REPO_ROOT = Path.cwd()
# Hentikan notebook kalau ternyata masih bukan root repo
assert is_repo_root(REPO_ROOT), f"{REPO_ROOT} bukan root repo thesis-2803 (src/core/config.py tidak ada)"
# Tampilkan lokasi repo
print("Repo root:", REPO_ROOT)
# Tampilkan branch yang sedang aktif
print("Branch   :", subprocess.run("git rev-parse --abbrev-ref HEAD", shell=True, capture_output=True, text=True).stdout.strip())

# 2) dependency (torch, pandas, networkx, tqdm sudah ada di Kaggle -> jangan di-upgrade;
#    `-U pandas` sebelumnya menarik pandas 3.x dan bentrok dengan dependency image Kaggle).
#    nltk dibutuhkan process_sql.py milik Spider.
# Install library yang belum tersedia di Kaggle
sh("pip install -q -U 'bitsandbytes>=0.46.1' sentence-transformers transformers nltk")

# 3) evaluator resmi Spider sudah ter-commit di external/spider_eval/ (JANGAN diedit) -> cukup dicek ada
# Periksa kedua file evaluator resmi satu per satu
for fname in ("evaluation.py", "process_sql.py"):
    # Hentikan notebook kalau filenya tidak ada
    assert (REPO_ROOT / "external" / "spider_eval" / fname).exists(), f"external/spider_eval/{fname} tidak ada di repo"

# 4) tokenizer data untuk parser SQL Spider
# nltk dipakai parser Spider untuk memecah SQL menjadi token
import nltk
# Unduh dua paket data tokenizer yang dibutuhkan
for pkg in ("punkt", "punkt_tab"):
    # Coba unduh paketnya
    try:
        # quiet=True supaya tidak banyak output
        nltk.download(pkg, quiet=True)
    # Kalau gagal, beri peringatan tanpa menghentikan notebook
    except Exception as e:
        # Tampilkan paket yang gagal dan alasannya
        print(f"[WARN] nltk.download({pkg}) gagal: {e}")
# Tanda bahwa seluruh setup sudah selesai
print("Setup selesai.")

## 2. Import

Semua modul dipanggil sebagai `src.<paket>.<modul>`. Yang dipakai notebook ini:

| Modul | Peran |
|---|---|
| `src.core.schema` | parse `tables.json` -> DataFrame -> graph NetworkX (node = kolom) |
| `src.retrieval.retrieval` | **GraphRAG**: two-stage linking, graph traversal, schema context (DDL) |
| `src.retrieval.baseline` | **Baseline**: graph per tabel, table-level linking |
| `src.generation.few_shot` | index embedding train set + retrieval contoh |
| `src.generation.generation` | prompt builder, LLM 4-bit, SQL cleaner |
| `src.experiments.pipeline` | `run_single()` (satu pertanyaan, GraphRAG) + helper raw_logs, confound diagnostics, dan `run_dimension_analysis()` (jembatan ke 6 dimensi) |
| `src.metrics.*`, `src.utils.*` | **evaluasi**: ESM/EX/CM (`esm_ex_cm`), gold schema (`sla`), normalisasi JOIN |

Sebagian helper di `pipeline.py` berawalan `_` (mis. `_column_nodes_to_predicted_schema`). Itu sengaja dipakai ulang supaya notebook **tidak menduplikasi** logika dan tidak bisa drift dari script CLI.

In [ ]:
# sys untuk path import, json untuk file JSON, re untuk mencari pola teks
import sys, json, re
# numpy untuk perhitungan angka dan pengambilan sampel acak
import numpy as np
# pandas untuk membuat tabel
import pandas as pd
# torch untuk menjalankan model di GPU
import torch
# Library untuk memuat model embedding BGE-M3
from sentence_transformers import SentenceTransformer

# Coba pakai progress bar tqdm
try:
    # Versi auto menyesuaikan tampilan dengan notebook
    from tqdm.auto import tqdm
# Kalau tqdm tidak ada, jalan terus tanpa progress bar
except ImportError:
    # Pengganti tqdm yang langsung mengembalikan datanya
    tqdm = lambda x, **kw: x

# Daftarkan root repo supaya modul src bisa di-import
sys.path.insert(0, str(REPO_ROOT))   # supaya `import src...` resolve

# Konfigurasi utama pipeline
from src.core.config import PipelineConfig
# Fungsi membaca schema Spider dan membangun graph per kolom
from src.core.schema import load_spider_schema, build_schema_graph

# Fungsi retrieval 2-Stage (GraphRAG)
from src.retrieval.retrieval import (
    # Membuat index embedding, lalu memilih tabel dan kolom
    build_schema_index, retrieve_candidate_tables, retrieve_candidate_columns,
    # Value linking, graph traversal, dan seluruh proses retrieval
    value_linked_columns, traversal_graph, trace_schema_paths, retrieve_graphrag_schema,
    # Membuat DDL, menghitung proxy, menghitung top-k, dan mengubah node menjadi nama kolom
    build_schema_context, evaluate_schema_linking, k_from_pct, column_nodes_to_schema,
)
# Fungsi retrieval 1-Stage ("baseline" di kode)
from src.retrieval.baseline import (
    # Wadah hasil, graph per tabel, dan index embedding tabel
    BaselineResult, build_table_graph, build_table_index,
    # Memilih tabel, melacak jalur antar tabel, dan membuat DDL
    semantic_linking_table_level, trace_table_paths, build_table_schema_context,
    # Menghitung proxy, menjalankan satu pertanyaan, dan mengubah node menjadi nama kolom
    evaluate_table_linking, run_single_baseline, table_nodes_to_schema,
)
# Fungsi membangun index few-shot dan mengambil contoh
from src.generation.few_shot import build_few_shot_index, build_few_shot_block
# Fungsi memuat LLM, menyusun prompt, dan menghasilkan SQL
from src.generation.generation import load_model_and_tokenizer, build_prompt, generate_sql_with_token_count

# Fungsi orkestrasi dari pipeline.py
from src.experiments.pipeline import (
    # Menjalankan satu pertanyaan 2-Stage dan mengatur seed
    run_single, set_seed, PipelineResult, build_gold_schema_context,
    # Ringkasan proxy, analisis metrik, analisis dimensi, dan folder per model
    print_schema_linking_summary, run_metrics_analysis, run_dimension_analysis, model_run_dirs,
    # Mengubah node hasil retrieval menjadi daftar "tabel.kolom"
    _column_nodes_to_predicted_schema, _table_nodes_to_predicted_schema,
    # Penghitung confound diagnostics
    _new_confound_diag, _update_confound_diag, _format_confound_diag,
)

# Modul pembuat tabel hasil, sama dengan yang dipakai Excel
from src.experiments import result_tables as rt
# Fungsi untuk menampilkan tabel di notebook
from src.experiments.result_tables import show

# Evaluator ESM, EX, dan CM per query
from src.metrics import esm_ex_cm
# Fungsi menghitung SLA dan mengambil kolom yang dipakai gold SQL
from src.metrics.sla import compute_sla, extract_ground_truth_schema
# Fungsi membaca daftar kolom tiap database
from src.utils.schema_utils import load_db_schema
# Fungsi menyamakan bentuk JOIN sebelum diparse evaluator resmi
from src.utils.sql_normalize import normalize_sql_file_for_parsing

# Pastikan semua import berhasil dan cek ketersediaan GPU
print("Imports OK | CUDA available:", torch.cuda.is_available())

## 3. Konfigurasi

`PipelineConfig` (`src/core/config.py`) adalah **sumber kebenaran tunggal** — jangan hardcode nilai di tempat lain. Yang bisa diatur di notebook ini:

- `TABLE_THRESHOLD` / `COLUMN_THRESHOLD` — **satu-satunya pilihan retrieval**: skor cosine minimum tabel (Stage 1, dipakai GraphRAG **dan** Baseline) dan kolom (Stage 2, hanya GraphRAG). Default 0.60 / 0.70, dipilih dengan SLA di `graphrag_sla_trial.ipynb`.
- **Preprocessing GraphRAG (tetap, bukan pilihan)** — default `config.py`, ditetapkan dari trial SLA: (1) normalisasi nama, (2) embed nama natural Spider (`StuID` -> `student id`), (3) teks kolom tanpa prefix tabel, (4) n-gram 1-2 (unigram + bigram) dengan stopword `edge`, (5) tanpa batas top-k% (hanya threshold), (6) value linking teks (exact match, tanpa angka), (7) tabel tanpa PK disambung di graph traversal, (8) traversal `fk_only` (hanya kolom JOIN). Jangan diubah di notebook ini.
- `FEW_SHOT_SELECTION` — cara memilih contoh few-shot: `"question"` (cosine pertanyaan mentah) atau `"skeleton"` (cosine pertanyaan yang domain-nya di-mask, `src/skeleton/masking.py`; contoh dengan **struktur/intent** sama). Berlaku untuk **kedua** pipeline. Prompt tetap menampilkan pertanyaan + SQL asli contohnya.
- `SAMPLE_RATIO` — fraksi dev set. `0.05` = trial cepat (~51 query). **`1.0` = run final** untuk skripsi.
- `MODEL` — **satu-satunya yang diganti tiap orang**: `"3B"`, `"7B"`, atau `"14B"`. Hasil tiap model masuk ke `outputs/models/<model>/` dan `data/raw_logs/<model>/` supaya tidak saling menimpa.

Parameter penting: `baseline_top_k_tables_pct` (batas atas opsional untuk Baseline, `0` = tanpa batas, hanya threshold), `few_shot_k=3` (**dipakai GraphRAG DAN Baseline**), `token_output_weight=3.0` (α di T = T_in + α·T_out, fixed dari proposal), `temperature=0.0` (greedy, deterministik), `seed=42`.

**Output** ditulis relatif ke root repo. Untuk run trial (`SAMPLE_RATIO < 1.0`) semuanya diarahkan ke `outputs/trial/models/<model>/` supaya **tidak menimpa** hasil run final.

In [ ]:
# ===== BAGIAN YANG DIGANTI TIAP ORANG: hanya MODEL =====
# Model LLM yang dijalankan: "3B", "7B", atau "14B"
MODEL = "7B"
# Porsi dev set yang dipakai
SAMPLE_RATIO = 0.05      # 0.05 = trial cepat | 1.0 = FULL dev set (wajib untuk QVT & run final skripsi)

# Satu-satunya pilihan retrieval adalah dua threshold ini (TABLE_THRESHOLD juga dipakai Baseline) (default config.py = 0.60 / 0.70,
# dari graphrag_sla_trial.ipynb). Sisanya preprocessing tetap, lihat print di bawah.
TABLE_THRESHOLD  = 0.60   # Stage 1: skor cosine minimum sebuah TABEL
COLUMN_THRESHOLD = 0.70   # Stage 2: skor cosine minimum sebuah KOLOM
FEW_SHOT_SELECTION = "skeleton"   # "question" | "skeleton" (dipakai GraphRAG DAN Baseline)

# Buat konfigurasi dengan pilihan threshold dan few-shot di atas
cfg = PipelineConfig(table_similarity_threshold=TABLE_THRESHOLD,
                     # Threshold kolom untuk 2-Stage
                     semantic_similarity_threshold=COLUMN_THRESHOLD,
                     # Cara memilih contoh few-shot
                     few_shot_selection=FEW_SHOT_SELECTION)
# Ubah nama pendek ("14B") menjadi nama lengkap model
cfg.llm_model = cfg.resolve_llm_model(MODEL)
# Kalau bukan di Kaggle, timpa lokasi data Spider, mis.:
# cfg = PipelineConfig(data_path=Path("data/spider_data"))

# Hentikan notebook kalau dataset Spider tidak ditemukan
assert cfg.data_path.exists(), (
    # Baris pertama pesan error
    f"Data Spider tidak ditemukan di {cfg.data_path}. Di Kaggle, tambahkan dataset Spider "
    # Baris kedua pesan error
    "lewat 'Add Input', atau isi PipelineConfig(data_path=...) dengan lokasi yang benar."
)

# True kalau ini run penuh, False kalau run percobaan
IS_FULL = SAMPLE_RATIO >= 1.0
# Tentukan folder hasil, raw_logs, dan QVT khusus untuk model ini
ARTIFACT_DIR, RAW_LOGS_DIR, QVT_DIR = model_run_dirs(cfg.llm_model, out_root=Path("outputs") if IS_FULL else Path("outputs/trial"))
# Run percobaan menyimpan raw_logs di folder trial supaya tidak menimpa run penuh
if not IS_FULL:
    # Arahkan raw_logs ke dalam folder hasil trial
    RAW_LOGS_DIR = ARTIFACT_DIR / "raw_logs"
# Buat folder prediksi, tabel, dan raw_logs
for d in (ARTIFACT_DIR / "predictions", ARTIFACT_DIR / "tables", RAW_LOGS_DIR):
    # Tidak error kalau foldernya sudah ada
    d.mkdir(parents=True, exist_ok=True)

# Kunci semua sumber acak supaya hasil run bisa diulang
set_seed(cfg.seed)   # random, numpy, torch, cudnn deterministik -> run reprodusibel

# Susun tabel ringkasan konfigurasi run
run_config = pd.DataFrame([
    # Lokasi dataset
    ("Data path", str(cfg.data_path)),
    # Model embedding
    ("Embedding model", cfg.embedding_model),
    # Model LLM yang dipilih
    ("LLM model", cfg.llm_model),
    # Threshold tabel
    ("Table threshold (Stage 1, both pipelines)", cfg.table_similarity_threshold),
    # Threshold kolom
    ("Column threshold (Stage 2, GraphRAG)", cfg.semantic_similarity_threshold),
    # Apakah embedding memakai nama natural Spider
    ("Embed natural names", cfg.embed_natural_names),
    # Cara menangani stopword
    ("Query stopword mode", cfg.query_stopword_mode),
    # Batas top-k tabel dan kolom
    ("Top-k tables / columns pct (0 = no cap)", f"{cfg.top_k_tables_pct:.0%} / {cfg.top_k_columns_pct:.0%}"),
    # Batas top-k tabel pada 1-Stage
    ("Baseline top-k tables pct (0 = no cap)", f"{cfg.baseline_top_k_tables_pct:.0%}"),
    # Status value linking
    ("Value linking (text / numbers)", f"{cfg.use_value_linking} / {cfg.value_link_numbers}"),
    # Kolom jalur yang masuk prompt
    ("Traversal mode", cfg.traversal_mode),
    # Apakah tabel tanpa PK disambung di graph
    ("Connect tables without PK", cfg.connect_no_pk_tables),
    # Panjang n-gram maksimum
    ("max_ngram", cfg.max_ngram),
    # Jumlah contoh few-shot
    ("Few-shot k (both pipelines)", cfg.few_shot_k),
    # Cara memilih few-shot
    ("Few-shot selection (both pipelines)", cfg.few_shot_selection),
    # Bobot token output pada rumus T
    ("α (output-token weight)", cfg.token_output_weight),
    # Jenis run
    ("SAMPLE_RATIO", f"{SAMPLE_RATIO} ({'full run' if IS_FULL else 'trial, outputs/trial/'})"),
    # Folder hasil model ini
    ("Results directory", str(ARTIFACT_DIR)),
    # Folder raw_logs model ini
    ("raw_logs directory", str(RAW_LOGS_DIR)),
# Beri nama kolom dan jadikan semua nilainya teks
], columns=["Parameter", "Value"]).astype({"Value": str})
# Tampilkan tabel konfigurasi
show(run_config, "Run configuration")

## 4. Load Data (Spider Dev Set)

`dev.json` = 1034 pasang (question, gold SQL, db_id). Subsampling (kalau `SAMPLE_RATIO < 1.0`) **identik dengan `run_comparison()`**: pakai `np.random.default_rng(cfg.seed)` sehingga sampel yang sama dipakai kedua pipeline dan bisa direproduksi.

Gold SQL subset ditulis ke `gold_subset.sql` dalam format resmi Spider (`SQL<TAB>db_id`) — dipakai nanti untuk cross-check dengan `evaluation.py` resmi.

> `query_id` di raw_logs = `dev_{posisi+1:04d}` **dalam sampel ini**. Hanya pada run penuh posisi itu sama dengan posisi di `dev.json` (syarat QVT).

In [ ]:
# Buka file dev set Spider
with open(cfg.dev_json, encoding="utf-8") as f:
    # Baca semua pertanyaan dan gold SQL
    dev_data = json.load(f)

# Run percobaan hanya memakai sebagian data
if SAMPLE_RATIO < 1.0:
    # Hitung jumlah sampel, minimal 1
    n = max(1, int(len(dev_data) * SAMPLE_RATIO))
    # Buat pengacak dengan seed tetap supaya sampelnya selalu sama
    rng = np.random.default_rng(cfg.seed)
    # Pilih posisi data secara acak tanpa pengulangan
    indices = rng.choice(len(dev_data), size=n, replace=False)
    # Ambil data terpilih dengan urutan aslinya
    dev_subset = [dev_data[i] for i in sorted(indices)]
# Run penuh memakai semua data
else:
    # Pakai seluruh dev set
    dev_subset = dev_data

# Tentukan lokasi file gold SQL sesuai sampel
gold_path = ARTIFACT_DIR / "gold_subset.sql"
# Buka file untuk ditulis
with open(gold_path, "w", encoding="utf-8") as f:
    # Proses tiap pertanyaan
    for item in dev_subset:
        # Tulis "SQL<TAB>db_id", format yang diminta evaluator resmi
        f.write(item["query"].strip() + "\t" + item["db_id"] + "\n")

# Tampilkan ringkasan data yang dievaluasi
show(pd.DataFrame([
    # Total pertanyaan di dev set
    {"Item": "Dev set queries", "Value": str(len(dev_data))},
    # Jumlah dan persentase pertanyaan yang dipakai
    {"Item": "Evaluated queries", "Value": f"{len(dev_subset)} ({len(dev_subset) / len(dev_data):.1%})"},
    # Jumlah database di sampel
    {"Item": "Databases in sample", "Value": str(len({d['db_id'] for d in dev_subset}))},
    # Lokasi file gold SQL
    {"Item": "Gold subset file", "Value": str(gold_path)},
]), "Evaluation data")
# Tampilkan contoh pertanyaan pertama
show(pd.DataFrame([{k: dev_subset[0][k] for k in ("db_id", "question", "query")}]), "First sample")

## 5. Bangun Schema Graph

Dua graph dibangun dari `tables.json` dengan granularitas berbeda — **ini inti perbedaan struktural GraphRAG vs Baseline**:

**GraphRAG** — node per **kolom** (`{db}.{table}.{col}`), diproses `build_schema_graph()`:
- edge `belongs_to_pk`: kolom non-PK -> PK tabel yang sama
- edge `foreign_key`: kolom FK -> kolom PK tabel lain
- memungkinkan seleksi sampai level kolom
- dengan `connect_no_pk_tables=True`, traversal memakai **salinan** graph ini yang menambah edge `same_table` di tabel tanpa PK (tabel penghubung seperti `Has_Pet`), supaya JOIN lewat tabel itu bisa dilalui. Graph aslinya tidak berubah

**Baseline** — node per **tabel** (`{db}.{table}`), diproses `build_table_graph()`:
- edge `foreign_key` antar tabel
- retrieval berhenti di level tabel; semua kolom tabel terpilih masuk prompt

In [ ]:
# Baca tables.json menjadi tabel berisi semua kolom
schema_df = load_spider_schema(cfg.tables_json)
# Bangun graph per kolom untuk 2-Stage
col_graph = build_schema_graph(schema_df)          # GraphRAG: node = kolom
# Bangun graph per tabel untuk 1-Stage
tbl_graph = build_table_graph(schema_df)           # Baseline: node = tabel

# Fungsi kecil untuk menghitung jumlah edge per jenis
edge_types = lambda G: pd.Series([d.get("relation") or d.get("type") or "edge" for *_e, d in G.edges(data=True)]).value_counts()
# Tampilkan ukuran kedua graph
show(pd.DataFrame([
    # Baris graph 2-Stage: jumlah node dan edge
    {"Graph": "GraphRAG (node = column)", "Nodes": col_graph.number_of_nodes(), "Edges": col_graph.number_of_edges(),
     # Rincian jenis edge graph 2-Stage
     "Edge types": ", ".join(f"{k}: {v:,}" for k, v in edge_types(col_graph).items())},
    # Baris graph 1-Stage: jumlah node dan edge
    {"Graph": "Baseline (node = table)", "Nodes": tbl_graph.number_of_nodes(), "Edges": tbl_graph.number_of_edges(),
     # Rincian jenis edge graph 1-Stage
     "Edge types": ", ".join(f"{k}: {v:,}" for k, v in edge_types(tbl_graph).items())},
]), f"Schema graphs (schema DataFrame: {len(schema_df):,} rows)")

### 5b. Cek arah foreign key (data asli vs DDL Baseline)

FK dibaca langsung dari `tables.json` (`foreign_keys` = pasangan indeks kolom: **[kolom yang mereferensikan, kolom yang direferensikan]**), lalu dibandingkan dengan DDL yang ditulis Baseline ke prompt. Setiap baris `FOREIGN KEY` harus muncul di tabel yang **mereferensikan**, bukan di tabel yang direferensikan. Ganti `CEK_FK_DB` untuk DB lain.

In [ ]:
# Database yang diperiksa foreign key-nya
CEK_FK_DB = "pets_1"
# Ambil definisi database itu dari tables.json
_tj = next(d for d in json.loads(cfg.tables_json.read_text(encoding="utf-8")) if d["db_id"] == CEK_FK_DB)
# Fungsi kecil untuk mengubah indeks kolom menjadi "tabel.kolom"
_name = lambda i: f'{_tj["table_names_original"][_tj["column_names_original"][i][0]]}.{_tj["column_names_original"][i][1]}'
# Tampilkan pasangan foreign key dari data asli
show(pd.DataFrame([{"Referencing column": _name(src), "Referenced column": _name(tgt)} for src, tgt in _tj["foreign_keys"]]),
     # Judul tabel
     f"Foreign keys in tables.json ({CEK_FK_DB})")
# Judul untuk DDL di bawah
print("DDL Baseline (semua tabel DB ini):")
# Tampilkan DDL semua tabel database itu untuk dibandingkan
print(build_table_schema_context(tbl_graph, [n for n, d in tbl_graph.nodes(data=True) if d.get("database") == CEK_FK_DB]))

## 6. Load Model

Dua model di-load **sekali** dan dipakai bersama kedua pipeline (jadi perbedaan hasil murni dari retrieval, bukan model):

| Model | Kegunaan | Catatan |
|---|---|---|
| `BAAI/bge-m3` | schema linking **dan** few-shot retrieval | Jangan ganti salah satu tanpa ganti keduanya |
| `MODEL` di Bagian 3 (`3B`, `7B`, atau `14B`) | SQL generation | 4-bit NF4, greedy decode (`temperature=0`); pilihannya ada di `cfg.llm_model_frame` |

Loading LLM butuh beberapa menit dan ~6 GB VRAM — pastikan Accelerator = GPU (T4).

In [ ]:
# Tampilkan model embedding yang dimuat
print(f"Loading embedding model: {cfg.embedding_model}")
# Muat model embedding
embed_model = SentenceTransformer(cfg.embedding_model)

# Tampilkan LLM yang dimuat
print(f"Loading LLM: {cfg.llm_model} (4-bit) ...")
# Muat LLM dan tokenizer sesuai MODEL yang dipilih
llm, tokenizer = load_model_and_tokenizer(cfg)
# Tanda bahwa kedua model sudah siap
print("Model siap.")

## 7. Bangun Index

Semua embedding disimpan **di memori** (tanpa vector DB eksternal) dan dibangun **sekali** di awal supaya tidak dihitung ulang di dalam loop:

- **Schema index GraphRAG** (`SchemaIndex`, per `db_id`): embedding tiap tabel + tiap kolom. Dengan `embed_natural_names=True` yang di-embed nama anotasi Spider (`StuID` -> `student id`); DDL tetap nama asli
- **Value index GraphRAG** (kalau `use_value_linking`): nilai sel tiap DB dibaca sekali (`SELECT DISTINCT ... LIMIT`, read-only) saat query pertama DB itu, lalu di-cache
- **Table index Baseline** (`TableSchemaIndex`, per `db_id`): embedding tiap tabel berformat `"table <nama> columns <kol1> <kol2> ..."`
- **Few-shot index** (`FewShotIndex`, dipakai **kedua** pipeline): embedding seluruh `train_spider.json` (~7000 pertanyaan), atau skeleton-nya (`data/skeletons/train.json`) kalau `few_shot_selection="skeleton"`; skeleton pertanyaan dev dibaca dari `data/skeletons/dev.json`. Contoh **hanya diambil dari train set**, bukan dev/test -> bukan kebocoran data (Spider cross-domain: DB train dan dev tidak beririsan).

In [ ]:
# Ambil daftar database di sampel tanpa duplikat
unique_db_ids = list(dict.fromkeys(item["db_id"] for item in dev_subset))
# Tampilkan jumlah database
print(f"{len(unique_db_ids)} database unik di sampel ini")

# Buat embedding kolom tiap database untuk 2-Stage
col_cache = {db_id: build_schema_index(col_graph, db_id, embed_model, use_natural_names=cfg.embed_natural_names)
             # Ulangi untuk semua database, dengan progress bar
             for db_id in tqdm(unique_db_ids, desc="Index kolom (GraphRAG)")}
# Buat embedding tabel tiap database untuk 1-Stage
tbl_cache = {db_id: build_table_index(tbl_graph, db_id, embed_model, use_natural_names=cfg.embed_natural_names)
             # Ulangi untuk semua database, dengan progress bar
             for db_id in tqdm(unique_db_ids, desc="Index tabel (Baseline)")}

# Nilai awal index few-shot: belum ada
few_shot_idx = None
# Index few-shot hanya dibutuhkan kalau k lebih dari 0
if cfg.few_shot_k > 0:
    # Tampilkan status proses
    print("Building few-shot index dari train set ...")
    # Buat embedding contoh dari train set
    few_shot_idx = build_few_shot_index(cfg.train_json, embed_model, cfg)
    # Tampilkan jumlah contoh dan cara seleksinya
    print(f"Few-shot index: {len(few_shot_idx.examples)} contoh training (selection={few_shot_idx.selection})")
# Kalau k = 0, tidak ada contoh yang dipakai
else:
    # Tampilkan bahwa kedua pipeline berjalan tanpa contoh
    print("few_shot_k=0 -> zero-shot untuk kedua pipeline")

## 8. Setup Evaluasi

Sebelum menjalankan pipeline, siapkan alat evaluasinya. Ada **dua jalur evaluasi** dan penting untuk tahu bedanya:

| Jalur | Dipakai untuk | Kode |
|---|---|---|
| **In-process** (sumber angka utama) | ESM/EX/CM per query -> `raw_logs` -> 6 dimensi | `src/metrics/esm_ex_cm.py` (**custom**, membungkus parser & evaluator **resmi Spider**) |
| **Subprocess** (cross-check) | ESM/EX agregat | `external/spider_eval/evaluation.py` (**resmi Spider**, tidak diubah) |

Yang disiapkan sekali di sini (bukan per query, karena mahal):
- `db_schema` — schema per DB untuk menurunkan **gold schema** (kolom yang dibutuhkan gold SQL) -> dasar Schema Linking Accuracy (SLA)
- `kmaps` — peta foreign key per DB, dibutuhkan parser Spider
- `spider_schema_cache` — objek `Schema` parser Spider (butuh koneksi SQLite), dibuat lazy per `db_id`

**Definisi metrik singkat:**
- **ESM** — struktur SQL (parse tree) sama dengan gold. Di Spider, ESM ≡ EM.
- **EX** — hasil eksekusi di SQLite sama dengan gold (pakai `eval_exec_match()` resmi, order-sensitive).
- **CM** — kecocokan per klausa (SELECT, FROM, WHERE, GROUP BY, ORDER BY, ...), dipakai Dimensi 3.
- **Token Consumption** — T = T_in + α·T_out (α = 3.0).

In [ ]:
# Tampilkan status proses
print("Loading db_schema + foreign-key maps ...")
# Baca daftar kolom tiap database
db_schema = load_db_schema(str(cfg.tables_json))
# Baca peta foreign key tiap database
kmaps = esm_ex_cm.build_kmaps(str(cfg.tables_json))
# Tempat menyimpan objek schema parser supaya tidak dibuat berulang
spider_schema_cache = {}

# Fungsi pembantu untuk mendapatkan lokasi file SQLite sebuah database
def db_path_of(db_id: str) -> str:
    # Susun path file .sqlite
    return str(cfg.db_dir / db_id / f"{db_id}.sqlite")

# Fungsi pembantu untuk mengevaluasi satu prediksi
def evaluate_pred(pred_sql: str, gold_sql: str, db_id: str):
    """Bungkus esm_ex_cm.evaluate_single_query(): 1 prediksi -> objek hasil (esm, ex, cm_per_clause, difficulty)."""
    # Ambil lokasi database
    path = db_path_of(db_id)
    # Buat objek schema kalau belum pernah dibuat
    if db_id not in spider_schema_cache:
        # Simpan supaya bisa dipakai lagi
        spider_schema_cache[db_id] = esm_ex_cm.build_schema_for_db(path)
    # Hitung ESM, EX, CM, dan tingkat kesulitan
    return esm_ex_cm.evaluate_single_query(pred_sql, gold_sql, db_id, path,
                                           # Schema dan peta foreign key database ini
                                           spider_schema_cache[db_id], kmaps[db_id])

# Tampilkan jumlah database yang berhasil dibaca
print(f"db_schema: {len(db_schema)} DB | kmaps: {len(kmaps)} DB")

## 9. Walk-through Satu Sampel: GraphRAG, Tahap demi Tahap

Sebelum loop penuh, kita bedah **satu pertanyaan** supaya jelas apa yang dikerjakan `run_single()`. Setiap sel di bawah = satu tahap pipeline. (Hasil di sini hanya demonstrasi dan **tidak dicatat** ke hasil akhir.)

Ganti `WALK_IDX` untuk melihat pertanyaan lain. Sel Stage 2 diakhiri `assert` yang memastikan langkah-langkah yang diprint **identik** dengan `retrieve_graphrag_schema()` yang dipakai loop utama.

In [ ]:
# Posisi pertanyaan yang ingin dibedah
WALK_IDX = 0
# Ambil pertanyaan pada posisi itu
item = dev_subset[WALK_IDX]
# Pisahkan pertanyaan, gold SQL, dan nama database
q, gold, db = item["question"], item["query"], item["db_id"]
# Tampilkan nama database
print(f"DB       : {db}")
# Tampilkan pertanyaan
print(f"Question : {q}")
# Tampilkan gold SQL
print(f"Gold SQL : {gold}")

# ---- [1] STAGE 1: pilih TABEL kandidat -------------------------------------
# Pertanyaan dipecah jadi n-gram (unigram + bigram; mode stopword cfg.query_stopword_mode), tiap n-gram
# di-embed dengan BGE-M3, lalu dibandingkan (cosine) ke embedding tiap tabel. Skor tabel = skor n-gram
# terbaiknya (max-pool). Tabel lolos kalau skornya >= table_similarity_threshold (dan, kalau
# top_k_tables_pct > 0, masuk top-k%). Kalau tidak ada yang lolos, diambil 1 tabel terbaik.
# Pilih tabel kandidat
cand_tables = retrieve_candidate_tables(q, col_cache[db], embed_model, cfg)
# Tampilkan threshold dan batas yang dipakai
print(f"\n[Stage 1] threshold tabel {cfg.table_similarity_threshold}, top-k% {cfg.top_k_tables_pct:.0%} "
      # Tampilkan tabel yang terpilih
      f"-> {len(cand_tables)} dari {len(col_cache[db].tables)} tabel: {cand_tables}")

In [ ]:
# ---- [2] STAGE 2: pilih KOLOM di dalam tabel kandidat ----------------------
# Tiap kolom diskor dengan n-gram terbaiknya (max-pool); kolom di bawah semantic_similarity_threshold
# dibuang (cap top_k_columns_pct kalau > 0). Kalau tidak ada yang lolos, diambil 1 kolom terbaik.
# Pilih kolom dari tabel kandidat
detected_cols = retrieve_candidate_columns(q, col_cache[db], cand_tables, embed_model, cfg)
# Tampilkan threshold dan jumlah kolom terpilih
print(f"[Stage 2] threshold kolom {cfg.semantic_similarity_threshold} -> {len(detected_cols)} kolom: "
      # Tampilkan nama kolomnya
      f"{[n.split('.', 1)[1] for n in detected_cols]}")

# ---- [2b] VALUE LINKING: n-gram pertanyaan yang PERSIS sama dengan nilai sel DB ----
# ("cat" -> pets.pettype). Kolomnya ditambahkan sebagai anchor, walaupun tabelnya tidak lolos Stage 1.
# Cari kolom dari nilai sel yang disebut di pertanyaan
linked = value_linked_columns(col_graph, db, q, cfg) if cfg.use_value_linking else []
# Gabungkan kolom Stage 2 dan value linking tanpa duplikat
anchors = list(dict.fromkeys(detected_cols + linked))
# Tampilkan kolom hasil value linking
print(f"[Value linking] {[n.split('.', 1)[1] for n in linked] if cfg.use_value_linking else '(mati)'}")

# ---- [3] GRAPH TRAVERSAL: cari jalur JOIN antar anchor ---------------------
# Untuk tiap pasang anchor dicari shortest path di graph skema (salinan dengan edge same_table kalau
# connect_no_pk_tables). traversal_mode="fk_only": dari jalur itu hanya kolom JOIN (kedua ujung edge
# foreign_key) yang ditambahkan; "full": semua node di jalur.
# Cari jalur terpendek antar kolom anchor
c_nodes, paths, relations = trace_schema_paths(traversal_graph(col_graph, cfg), db, anchors)
# Mode fk_only hanya mengambil kolom JOIN
if cfg.traversal_mode == "fk_only":
    # Ambil kedua ujung setiap edge foreign key
    path_nodes = [n for r in relations if r["relation"] == "foreign_key" for n in (r["from"], r["to"])]
# Mode full mengambil semua kolom di jalur
else:
    # Ambil semua node pada setiap jalur
    path_nodes = [n for p in paths for n in p]
# Gabungkan anchor dan kolom jalur tanpa duplikat
column_nodes = list(dict.fromkeys(c_nodes + path_nodes))
# Tampilkan jumlah anchor dan jalur
print(f"\n[Traversal] {len(c_nodes)} anchor, {len(paths)} jalur, mode={cfg.traversal_mode}")
# Tampilkan maksimal 5 jalur
for p in paths[:5]:
    # Tulis jalur dalam bentuk kolom -> kolom
    print("   ", " -> ".join(n.split(".", 1)[1] for n in p))
# Tampilkan kolom tambahan dari traversal
print(f"[Traversal] ditambahkan: {[n.split('.', 1)[1] for n in column_nodes if n not in c_nodes]}")
# Tampilkan total kolom yang akan masuk prompt
print(f"[Traversal] total kolom yang masuk konteks: {len(column_nodes)}")

# Langkah [1]-[3] di atas = retrieve_graphrag_schema(), fungsi yang dipanggil run_single() di loop utama.
# Pastikan walk-through ini sama dengan fungsi yang dipakai loop utama
assert column_nodes == retrieve_graphrag_schema(col_graph, db, q, col_cache[db], embed_model, cfg)[0], \
    "walk-through tidak sama dengan retrieve_graphrag_schema()"

In [ ]:
# ---- [4] SCHEMA CONTEXT: bangun DDL dari kolom terpilih -------------------
# Ubah kolom terpilih menjadi teks CREATE TABLE
schema_context = build_schema_context(col_graph, column_nodes)
# Judul untuk DDL di bawah
print("[Schema context yang dikirim ke LLM]")
# Tampilkan DDL
print(schema_context)

# Evaluasi retrieval (proxy internal, BUKAN SLA resmi — SLA dihitung nanti dari raw_logs via sla.py)
# Hitung recall dan precision proxy
recall, precision = evaluate_schema_linking(gold, column_nodes, col_graph, db)
# Tampilkan nilai proxy
print(f"\nProxy schema-linking: recall={recall:.2f}  precision={precision:.2f}")
# Hitung SLA kolom dengan membandingkan kolom gold dan kolom terpilih
g_sla = compute_sla(set(extract_ground_truth_schema(gold, db, db_schema)), set(column_nodes_to_schema(col_graph, column_nodes)))
# Tampilkan nilai SLA
print(f"SLA kolom (metrik tahap 1): P={g_sla.precision:.2f} R={g_sla.recall:.2f} F1={g_sla.f1:.2f}")

# ---- [5] FEW-SHOT: contoh mirip dari train_spider.json --------------------
# build_few_shot_block() = retrieve + format; helper yang sama dipakai Baseline -> contoh identik.
# few_shot_selection="skeleton": yang dibandingkan adalah skeleton (domain di-mask), bukan pertanyaan mentah.
# Cek apakah few-shot aktif dan memakai skeleton
if few_shot_idx is not None and few_shot_idx.selection == "skeleton":
    # Tampilkan skeleton pertanyaan ini
    print(f"[Skeleton] {few_shot_idx.query_skeletons.get(q)}")
# Ambil contoh few-shot dan susun menjadi teks
few_shot_block = build_few_shot_block(q, few_shot_idx, embed_model, cfg)
# Judul untuk contoh di bawah
print("\n[Few-shot block]")
# Tampilkan contoh, atau keterangan kalau kosong
print(few_shot_block if few_shot_block else "(kosong: few_shot_k=0)")

In [ ]:
# ---- [6] PROMPT BUILDER ---------------------------------------------------
# Format hybrid: Task / Database Schema / Examples / Question / Instructions / Answer.
# Angka & string literal yang muncul di pertanyaan diekstrak dan disertakan supaya LLM memakai nilai persis.
# Kumpulkan nilai yang disebut di pertanyaan
extracted_values = {
    # Teks di dalam tanda kutip
    "strings": re.findall(r"'([^']*)'", q),
    # Angka
    "numbers": re.findall(r"\d+", q),
}
# Susun prompt lengkap untuk LLM
prompt = build_prompt(q, schema_context, extracted_values, few_shot_block)
# Hitung jumlah token prompt (T_in)
n_in = len(tokenizer.encode(prompt))
# Tampilkan prompt
print(prompt)
# Tampilkan jumlah token prompt
print(f"\n[T_in] prompt = {n_in} token")

# ---- [7] LLM + [8] SQL CLEANER --------------------------------------------
# Qwen2.5-Coder 4-bit, greedy. generate_sql_with_token_count() juga menghitung T_out
# dari output MENTAH sebelum dibersihkan, lalu SQL cleaner membuang markdown/alias aneh.
# Hasilkan SQL dan hitung token output (T_out)
walk_pred, n_out = generate_sql_with_token_count(prompt, llm, tokenizer, cfg)
# Tampilkan T_out dan total konsumsi token
print(f"\n[T_out] {n_out} token   ->   T = T_in + {cfg.token_output_weight} x T_out = {n_in + cfg.token_output_weight * n_out:.0f}")
# Tampilkan SQL prediksi
print(f"Pred SQL : {walk_pred}")
# Tampilkan gold SQL sebagai pembanding
print(f"Gold SQL : {gold}")

### 9b. Evaluasi sampel yang sama (in-process)

`evaluate_single_query()` menjalankan tiga hal sekaligus atas pasangan (prediksi, gold): **ESM** (bandingkan parse tree), **EX** (eksekusi kedua SQL di SQLite), **CM per klausa**, plus **difficulty** yang diturunkan dari gold SQL (easy/medium/hard/extra, pakai `eval_hardness()` resmi Spider).

In [ ]:
# Hitung ESM, EX, CM, dan tingkat kesulitan untuk prediksi ini
walk_eval = evaluate_pred(walk_pred, gold, db)
# Ambil kolom yang dipakai gold SQL
walk_gold_schema = sorted(extract_ground_truth_schema(gold, db, db_schema))
# Tampilkan hasil evaluasi sampel ini
show(pd.DataFrame([
    # Tingkat kesulitan
    {"Field": "Difficulty", "Value": walk_eval.difficulty},
    # Hasil ESM (1 = cocok)
    {"Field": "ESM", "Value": walk_eval.esm},
    # Hasil EX (1 = cocok)
    {"Field": "EX", "Value": walk_eval.ex},
    # Kolom yang dipakai gold SQL
    {"Field": "Gold schema (columns used by gold SQL)", "Value": ", ".join(walk_gold_schema)},
    # Kolom yang dikirim 2-Stage ke LLM
    {"Field": "Predicted schema (columns sent by GraphRAG)", "Value": ", ".join(_column_nodes_to_predicted_schema(col_graph, column_nodes))},
# Jadikan semua nilai teks, lalu beri judul tabel
]).astype({"Value": str}), "Walk-through sample: GraphRAG evaluation")
# Tampilkan kecocokan per klausa SQL
show(pd.DataFrame([{rt.CLAUSE_LABEL.get(k, k): v for k, v in walk_eval.cm_per_clause.items()}]), "Component match per clause (1 = match)")

## 10. Walk-through Satu Sampel: Baseline

Pertanyaan yang sama, sekarang lewat Baseline. Bedanya ada di **granularitas**: yang dipilih adalah **tabel utuh**, dan **semua kolom** tabel itu masuk prompt. Few-shot-nya **sama persis** dengan GraphRAG (`cfg.few_shot_k` contoh dari index yang sama), jadi selisih hasil murni dari retrieval.

In [ ]:
# ---- [1] Table-level linking: SAMA dengan Stage 1 GraphRAG (n-gram, skor tabel = n-gram terbaik, ----
# ---- lolos kalau >= table_similarity_threshold), bedanya yang dicocokkan teks tabel "table <nama> columns <kolom...>"
# Pilih tabel yang relevan dengan pertanyaan
detected_tables = semantic_linking_table_level(tbl_cache[db], q, embed_model, cfg)
# Tampilkan threshold dan batas yang dipakai
print(f"[Linking] threshold tabel {cfg.table_similarity_threshold}, batas atas {cfg.baseline_top_k_tables_pct:.0%} "
      # Tampilkan tabel yang terpilih
      f"-> {len(detected_tables)} dari {len(tbl_cache[db].table_node_ids)} tabel: {detected_tables}")

# ---- [2] Table path tracing: tambahkan tabel perantara via FK bila ada gap konektivitas ----
# Tambahkan tabel penghubung yang dibutuhkan untuk JOIN
table_nodes = trace_table_paths(tbl_graph, detected_tables)
# Tampilkan daftar tabel akhir
print(f"[Path tracing] tabel final: {table_nodes}")

# ---- [3] Schema context: DDL dengan SEMUA kolom tiap tabel (tanpa pruning) ----
# Ubah tabel terpilih menjadi teks CREATE TABLE
bl_context = build_table_schema_context(tbl_graph, table_nodes)
# Judul untuk DDL di bawah
print("\n[Schema context yang dikirim ke LLM]")
# Tampilkan DDL
print(bl_context)

# Hitung recall dan precision proxy
bl_recall, bl_precision = evaluate_table_linking(gold, table_nodes, tbl_graph, db)
# Tampilkan nilai proxy
print(f"\nProxy schema-linking: recall={bl_recall:.2f}  precision={bl_precision:.2f}")
# Hitung SLA kolom dengan membandingkan kolom gold dan semua kolom tabel terpilih
b_sla = compute_sla(set(extract_ground_truth_schema(gold, db, db_schema)), set(table_nodes_to_schema(tbl_graph, table_nodes)))
# Tampilkan nilai SLA
print(f"SLA kolom (metrik tahap 1): P={b_sla.precision:.2f} R={b_sla.recall:.2f} F1={b_sla.f1:.2f}")

# ---- [4] Prompt (few-shot SAMA dengan GraphRAG) -> LLM -> cleaner ----
# Susun prompt dengan few-shot yang sama
bl_prompt = build_prompt(q, bl_context, extracted_values, few_shot_block)
# Hitung jumlah token prompt
bl_n_in = len(tokenizer.encode(bl_prompt))
# Hasilkan SQL dan hitung token output
bl_pred, bl_n_out = generate_sql_with_token_count(bl_prompt, llm, tokenizer, cfg)
# Hitung ESM, EX, dan CM
bl_eval = evaluate_pred(bl_pred, gold, db)
# Tampilkan SQL prediksi
print(f"Pred SQL : {bl_pred}")
# Tampilkan perbandingan kedua varian untuk sampel ini
show(pd.DataFrame([
    # Baris 2-Stage: SQL, token, ESM, dan EX
    {"Pipeline": "GraphRAG", "Predicted SQL": walk_pred, "T_in": n_in, "T_out": n_out, "ESM": walk_eval.esm, "EX": walk_eval.ex,
     # Baris 2-Stage: SLA
     "SLA column P": g_sla.precision, "SLA column R": g_sla.recall},
    # Baris 1-Stage: SQL, token, ESM, dan EX
    {"Pipeline": "Baseline", "Predicted SQL": bl_pred, "T_in": bl_n_in, "T_out": bl_n_out, "ESM": bl_eval.esm, "EX": bl_eval.ex,
     # Baris 1-Stage: SLA
     "SLA column P": b_sla.precision, "SLA column R": b_sla.recall},
# Format SLA sebagai persen, lalu beri judul tabel
]).pipe(rt.with_formats, {"SLA column P": "pct", "SLA column R": "pct"}), "Walk-through sample: GraphRAG vs Baseline")

## 11. Loop Utama — Kedua Pipeline atas Semua Sampel

Loop ini **setara dengan `run_comparison()`** di `pipeline.py`, hanya ditulis eksplisit. Untuk tiap pertanyaan:

1. `run_single()` -> prediksi GraphRAG (+ recall/precision proxy, kolom terpilih, T_in, T_out)
2. `run_single_baseline()` -> prediksi Baseline (+ tabel terpilih, T_in, T_out), few-shot index & k yang sama
3. Evaluasi in-process kedua prediksi (ESM/EX/CM/difficulty) + turunkan **gold schema**
4. Tambahkan satu entri ke `raw_logs` tiap kondisi (skema 12 field, lihat `context/EVALUATION_ANALYSIS_GUIDE.md` Bagian 2)
5. Update **confound diagnostics** (read-only; tidak mengubah nilai metrik apa pun)

**Penanganan error** (penting untuk run panjang di Kaggle) — dua `try/except` sengaja terpisah:
- gagal saat *generate* -> fallback `"SELECT 1"` dan loop lanjut (tidak ada satu error pun yang mematikan run berjam-jam)
- gagal saat *evaluasi* -> prediksi tetap tersimpan, hanya entri raw_log query itu yang dilewati; jumlahnya dilaporkan di akhir

In [ ]:
# Tempat hasil ringkas tiap varian
graphrag_results, baseline_results = [], []
# Tempat catatan lengkap per pertanyaan (raw_logs)
graphrag_raw_logs, baseline_raw_logs = [], []
# Tempat teks SQL prediksi
graphrag_preds, baseline_preds = [], []
# Siapkan penghitung confound diagnostics
confound_diag = _new_confound_diag()
# Tempat mencatat error generate dan evaluasi
gen_errors, eval_errors = [], []
# Lebar garis pemisah pada progres
W = 72

# Proses tiap pertanyaan dengan progress bar
for i, item in enumerate(tqdm(dev_subset, desc="GraphRAG + Baseline")):
    # Ambil database, pertanyaan, dan gold SQL
    db_id, question, gold_sql = item["db_id"], item["question"], item["query"]

    # ---- GraphRAG ----
    # Coba jalankan 2-Stage
    try:
        # Terima SQL, proxy, schema, kolom terpilih, dan jumlah token
        g_pred, g_recall, g_prec, g_schema, g_column_nodes, g_n_in, g_n_out = run_single(
            # Input pertanyaan, graph, dan model
            question, gold_sql, db_id, col_graph, embed_model, llm, tokenizer, cfg,
            # Index kolom dan index few-shot yang sudah dibuat
            schema_index=col_cache[db_id], few_shot_index=few_shot_idx,
        )
    # Kalau generate gagal, catat errornya dan lanjut
    except Exception as e:
        # Catat varian, posisi, database, dan pesan error
        gen_errors.append(("graphrag", i, db_id, repr(e)))
        # Pakai SQL pengganti supaya run tidak berhenti
        g_pred, g_recall, g_prec, g_schema = "SELECT 1", 0.0, 0.0, ""
        # Kosongkan kolom dan token
        g_column_nodes, g_n_in, g_n_out = [], 0, 0

    # ---- Baseline ----
    # Coba jalankan 1-Stage
    try:
        # Terima SQL, proxy, tabel terpilih, dan jumlah token
        b_pred, b_recall, b_prec, b_table_nodes, b_n_in, b_n_out = run_single_baseline(
            # Input pertanyaan, graph, dan model
            question, gold_sql, db_id, tbl_graph, embed_model, llm, tokenizer, cfg,
            # Index tabel dan index few-shot yang sama
            table_index=tbl_cache[db_id], few_shot_index=few_shot_idx,
        )
    # Kalau generate gagal, catat errornya dan lanjut
    except Exception as e:
        # Catat varian, posisi, database, dan pesan error
        gen_errors.append(("baseline", i, db_id, repr(e)))
        # Pakai SQL pengganti supaya run tidak berhenti
        b_pred, b_recall, b_prec = "SELECT 1", 0.0, 0.0
        # Kosongkan tabel dan token
        b_table_nodes, b_n_in, b_n_out = [], 0, 0

    # Simpan hasil ringkas 2-Stage
    graphrag_results.append(PipelineResult(index=i + 1, db_id=db_id, question=question, gold_sql=gold_sql,
                                           # SQL prediksi dan nilai proxy
                                           pred_sql=g_pred, recall=g_recall, precision=g_prec))
    # Simpan hasil ringkas 1-Stage
    baseline_results.append(BaselineResult(index=i + 1, db_id=db_id, question=question, gold_sql=gold_sql,
                                           # SQL prediksi dan nilai proxy
                                           pred_sql=b_pred, recall=b_recall, precision=b_prec))
    # Simpan teks SQL 2-Stage
    graphrag_preds.append(g_pred.strip())
    # Simpan teks SQL 1-Stage
    baseline_preds.append(b_pred.strip())

    # ---- Evaluasi in-process + raw_logs (try/except TERPISAH dari generation) ----
    # Coba evaluasi kedua prediksi
    try:
        # Ambil kolom yang dipakai gold SQL
        gold_schema = sorted(extract_ground_truth_schema(gold_sql, db_id, db_schema))
        # Evaluasi prediksi 2-Stage
        g_eval = evaluate_pred(g_pred, gold_sql, db_id)
        # Evaluasi prediksi 1-Stage
        b_eval = evaluate_pred(b_pred, gold_sql, db_id)

        # instrumentasi read-only (tidak mengubah nilai raw_logs di bawah)
        # Perbarui penghitung confound
        _update_confound_diag(confound_diag, db_path_of(db_id), gold_sql, g_pred, g_eval, b_pred, b_eval)

        # Ambil tingkat kesulitan pertanyaan
        difficulty = g_eval.difficulty          # properti gold_sql -> g_eval & b_eval selalu sama
        # Buat ID pertanyaan dari posisinya
        query_id = f"dev_{i + 1:04d}"
        # Simpan catatan lengkap 2-Stage
        graphrag_raw_logs.append({
            # Identitas pertanyaan
            "query_id": query_id, "db_id": db_id, "difficulty": difficulty,
            # SQL gold dan prediksi
            "gold_sql": gold_sql, "predicted_sql": g_pred,
            # Kolom gold
            "gold_schema": gold_schema,
            # Kolom yang dikirim ke LLM
            "predicted_schema": _column_nodes_to_predicted_schema(col_graph, g_column_nodes),
            # Jumlah token input dan output
            "token_input": g_n_in, "token_output": g_n_out,
            # Hasil ESM dan EX
            "esm_result": g_eval.esm, "ex_result": g_eval.ex,
            # Hasil CM per klausa
            "cm_per_clause": g_eval.cm_per_clause,
        })
        # Simpan catatan lengkap 1-Stage
        baseline_raw_logs.append({
            # Identitas pertanyaan
            "query_id": query_id, "db_id": db_id, "difficulty": difficulty,
            # SQL gold dan prediksi
            "gold_sql": gold_sql, "predicted_sql": b_pred,
            # Kolom gold
            "gold_schema": gold_schema,
            # Kolom yang dikirim ke LLM
            "predicted_schema": _table_nodes_to_predicted_schema(tbl_graph, b_table_nodes),
            # Jumlah token input dan output
            "token_input": b_n_in, "token_output": b_n_out,
            # Hasil ESM dan EX
            "esm_result": b_eval.esm, "ex_result": b_eval.ex,
            # Hasil CM per klausa
            "cm_per_clause": b_eval.cm_per_clause,
        })
    # Kalau evaluasi gagal, prediksi tetap tersimpan
    except Exception as e:
        # Catat posisi, database, dan pesan error
        eval_errors.append((i, db_id, repr(e)))

    # Tampilkan progres pada pertanyaan pertama, tiap 20 pertanyaan, dan pertanyaan terakhir
    if i == 0 or (i + 1) % 20 == 0 or (i + 1) == len(dev_subset):
        # Garis pembuka
        print("\n" + "=" * W)
        # Nomor pertanyaan dan database
        print(f"  [{i + 1:>4}/{len(dev_subset)}]  DB: {db_id}")
        # Garis pemisah
        print("=" * W)
        # Pertanyaan
        print(f"  Q              : {question}")
        # Gold SQL
        print(f"  GOLD           : {gold_sql}")
        # Prediksi dan proxy 2-Stage
        print(f"  [GraphRAG] pred: {g_pred}   (R={g_recall*100:.0f}% P={g_prec*100:.0f}%)")
        # Prediksi dan proxy 1-Stage
        print(f"  [Baseline] pred: {b_pred}   (R={b_recall*100:.0f}% P={b_prec*100:.0f}%)")

# Tampilkan ringkasan loop setelah selesai
show(pd.DataFrame([
    # Jumlah pertanyaan yang diproses
    {"Item": "Samples processed", "Value": len(graphrag_preds)},
    # Jumlah catatan raw_logs
    {"Item": "raw_logs entries (per pipeline)", "Value": len(graphrag_raw_logs)},
    # Jumlah error saat generate
    {"Item": "Generation errors", "Value": len(gen_errors)},
    # Jumlah error saat evaluasi
    {"Item": "Evaluation errors", "Value": len(eval_errors)},
]), "Main loop")
# Kalau ada error, tampilkan rinciannya
if gen_errors or eval_errors:
    # Rincian error generate
    show(pd.DataFrame([{"Stage": "generation", "Pipeline": p, "Index": i, "DB": d, "Error": e} for p, i, d, e in gen_errors]
                      # Ditambah rincian error evaluasi
                      + [{"Stage": "evaluation", "Pipeline": "both", "Index": i, "DB": d, "Error": e} for i, d, e in eval_errors]),
         # Judul tabel
         "Errors")

## 12. Simpan Artefak

- `predictions.txt` / `baseline_predictions.txt` — satu SQL per baris, urutan = urutan `gold_subset.sql`
- `raw_logs/{graphrag,baseline}_log.json` — **input untuk semua dimensi analisis**
- `comparison_report.txt` — ringkasan retrieval + blok CONFOUND DIAGNOSTICS (teks)
- `confound_diagnostics.json` — counter confound yang sama, dibaca export Excel

Ada pengecekan jumlah baris karena `evaluation.py` mencocokkan gold dan prediksi **berdasarkan nomor baris**; satu baris meleset = seluruh skor salah.

> Kalau ada query yang gagal dievaluasi, `raw_logs` lebih pendek dari sampel. Untuk run final sebaiknya **0 error** — investigasi dulu sebelum lanjut ke QVT/dimensi.

In [ ]:
# Lokasi file prediksi 2-Stage
g_pred_path = ARTIFACT_DIR / "predictions" / "predictions.txt"
# Lokasi file prediksi 1-Stage
b_pred_path = ARTIFACT_DIR / "predictions" / "baseline_predictions.txt"
# Tulis prediksi 2-Stage, satu SQL per baris
g_pred_path.write_text("\n".join(graphrag_preds) + "\n", encoding="utf-8")
# Tulis prediksi 1-Stage, satu SQL per baris
b_pred_path.write_text("\n".join(baseline_preds) + "\n", encoding="utf-8")

# Tempat hasil pengecekan jumlah baris
line_check = []
# Periksa kedua file prediksi
for label, path in (("GraphRAG", g_pred_path), ("Baseline", b_pred_path)):
    # Hitung baris yang tidak kosong
    n_lines = len([l for l in path.read_text(encoding="utf-8").split("\n") if l.strip()])
    # Simpan hasil pengecekan
    line_check.append({"Pipeline": label, "File": str(path), "Prediction lines": n_lines, "Samples": len(dev_subset),
                       # True kalau jumlah baris sama dengan jumlah sampel
                       "Lines = samples": n_lines == len(dev_subset)})
# Tampilkan hasil pengecekan
show(pd.DataFrame(line_check), "Prediction files (evaluation.py matches gold and prediction by line number)")

# Buka file raw_logs 2-Stage
with open(RAW_LOGS_DIR / "graphrag_log.json", "w", encoding="utf-8") as f:
    # Simpan catatan per pertanyaan
    json.dump(graphrag_raw_logs, f, ensure_ascii=False, indent=2)
# Buka file raw_logs 1-Stage
with open(RAW_LOGS_DIR / "baseline_log.json", "w", encoding="utf-8") as f:
    # Simpan catatan per pertanyaan
    json.dump(baseline_raw_logs, f, ensure_ascii=False, indent=2)
# Tampilkan lokasi dan jumlah catatan
print(f"raw_logs -> {RAW_LOGS_DIR}/ ({len(graphrag_raw_logs)} graphrag, {len(baseline_raw_logs)} baseline)")

# Proxy recall/precision (name-matching internal, BUKAN SLA; SLA resmi ada di Bagian 13)
# Susun tabel nilai proxy
proxy = pd.DataFrame([
    # Baris recall: nilai 1-Stage
    {"Metric": "Recall (proxy)", "Baseline": np.mean([r.recall for r in baseline_results]),
     # Baris recall: nilai 2-Stage
     "GraphRAG": np.mean([r.recall for r in graphrag_results])},
    # Baris precision: nilai 1-Stage
    {"Metric": "Precision (proxy)", "Baseline": np.mean([r.precision for r in baseline_results]),
     # Baris precision: nilai 2-Stage
     "GraphRAG": np.mean([r.precision for r in graphrag_results])},
])
# Hitung selisih dalam poin persen
proxy["Δ (GraphRAG − Baseline)"] = (proxy["GraphRAG"] - proxy["Baseline"]) * 100
# Tampilkan tabel proxy dengan format persen
show(rt.with_formats(proxy, {"Baseline": "pct", "GraphRAG": "pct", "Δ (GraphRAG − Baseline)": "pp"}),
     # Judul tabel
     "Schema-linking proxy (name matching, not SLA)")
# Tampilkan tabel confound diagnostics
show(rt.confound_table(confound_diag))

# Ambil recall dan precision 2-Stage dalam persen
g_r, g_p = proxy.loc[0, "GraphRAG"] * 100, proxy.loc[1, "GraphRAG"] * 100
# Ambil recall dan precision 1-Stage dalam persen
b_r, b_p = proxy.loc[0, "Baseline"] * 100, proxy.loc[1, "Baseline"] * 100
# Susun isi laporan teks
report = "\n".join([
    # Baris kosong dan garis pembuka
    "", "=" * 70,
    # Judul laporan dan garis
    "  COMPARISON REPORT — GraphRAG (column/node) vs Baseline (table/node)", "=" * 70,
    # Nama kolom dan garis
    f"  {'Metric':<26} {'GraphRAG':>10}  {'Baseline':>10}  {'Delta (G-B)':>12}", "-" * 70,
    # Baris recall
    f"  {'Recall (proxy)':<26} {g_r:>9.2f}%  {b_r:>9.2f}%  {g_r-b_r:>+11.2f}%",
    # Baris precision
    f"  {'Precision (proxy)':<26} {g_p:>9.2f}%  {b_p:>9.2f}%  {g_p-b_p:>+11.2f}%",
    # Garis pemisah
    "-" * 70,
    # Catatan bahwa nilai ini bukan SLA
    "  (proxy = name-matching internal; SLA resmi dihitung dari raw_logs via sla.py)",
    # Garis penutup dan baris kosong
    "=" * 70, "",
# Tambahkan blok confound diagnostics di akhir laporan
]) + _format_confound_diag(confound_diag)
# Simpan laporan teks
(ARTIFACT_DIR / "tables" / "comparison_report.txt").write_text(report, encoding="utf-8")
# Simpan penghitung confound sebagai JSON
(ARTIFACT_DIR / "tables" / "confound_diagnostics.json").write_text(json.dumps(confound_diag, indent=2), encoding="utf-8")

## 13. Hasil 6 Metrik

Keenam metrik dihitung dari `raw_logs` model yang dipilih, lalu disimpan ke `metrics_results.json`. Urutannya mengikuti `context/EVALUATION_ANALYSIS_GUIDE.md` Bagian 1.

| No | Metrik | Pertanyaan yang dijawab |
|---|---|---|
| 1 | **SLA** | Seberapa tepat tabel dan kolom yang dikirim ke LLM? |
| 2 | **ESM** | Apakah struktur SQL sama dengan gold? |
| 3 | **EX** | Apakah hasil eksekusi SQL sama dengan gold? |
| 4 | **CM** | Klausa SQL mana yang cocok? |
| 5 | **Token + TEP** | Berapa token terpakai, dan sepadan tidak dengan perubahan EX? |
| 6 | **QVT** | Apakah jawaban tetap benar saat pertanyaan diparafrase? |

**1-Stage** adalah `baseline` di kode, dan **2-Stage** adalah `graphrag` di kode. QVT hanya tersedia pada run penuh (`SAMPLE_RATIO = 1.0`).

> Recall/precision proxy (Bagian 12) bukan SLA. Angka SLA yang dilaporkan ada di bagian ini.

In [ ]:
# Run penuh juga menyiapkan data QVT; lalu keenam metrik dihitung dari raw_logs dan disimpan
metrics_results = run_metrics_analysis(
    # Folder raw_logs model ini
    raw_logs_dir=RAW_LOGS_DIR,
    # metrics_results.json ditulis ke folder tabel model ini
    out_dir=ARTIFACT_DIR / "tables",
    # File dev set, dibutuhkan untuk menyusun pasangan parafrase QVT
    dev_json=cfg.dev_json,
    # QVT hanya dihitung pada run penuh
    full_dev=IS_FULL,
    # Konfigurasi run dicatat ke dalam hasil
    cfg=cfg,
    # Porsi data yang dipakai run ini
    sample_ratio=SAMPLE_RATIO,
    # Folder data QVT model ini
    qvt_dir=QVT_DIR,
)
# Ubah hasil metrik menjadi tabel, berurutan dari SLA sampai QVT
metric_groups = rt.metric_tables(metrics_results)

# Fungsi pembantu untuk menampilkan semua tabel dari satu metrik
def show_metric(position: int) -> None:
    # Ambil nama sheet, judul, dan tabel metrik itu
    _sheet, title, tables = metric_groups[position]
    # Tampilkan judul metrik
    print(title)
    # Proses tiap tabel metrik itu
    for table in tables:
        # Tampilkan tabelnya
        show(table)

# Beri tahu kalau ada pertanyaan yang gagal dievaluasi
if len(graphrag_raw_logs) != len(dev_subset):
    # Tampilkan jumlah catatan dan jumlah sampel
    print(f"[!] raw_logs ({len(graphrag_raw_logs)}) < sampel ({len(dev_subset)}): ada query yang gagal dievaluasi.")
# Tampilkan model dan lokasi file yang tersimpan
print(f"Model: {cfg.llm_model} | Tersimpan: {ARTIFACT_DIR / 'tables' / 'metrics_results.json'}")

### 13.1 Schema Linking Accuracy (SLA)

Precision, recall, dan F1 skema yang dikirim ke LLM dibandingkan dengan skema gold SQL. Nilainya dirata-ratakan per query.

In [ ]:
# Tampilkan metrik 1: SLA
show_metric(0)

### 13.2 Exact Set Match (ESM)

Persentase prediksi yang struktur SQL-nya sama dengan gold.

In [ ]:
# Tampilkan metrik 2: ESM
show_metric(1)

### 13.3 Execution Accuracy (EX)

Persentase prediksi yang hasil eksekusinya sama dengan gold.

In [ ]:
# Tampilkan metrik 3: EX
show_metric(2)

### 13.4 Component Match (CM)

Kecocokan per klausa SQL. Nilainya sengaja tidak digabung menjadi satu angka.

In [ ]:
# Tampilkan metrik 4: CM
show_metric(3)

### 13.5 Token Consumption dan TEP

T = T_in + α × T_out. TEP = (ΔEX / EX 1-Stage) / (ΔT / T 1-Stage), dengan Δ = 2-Stage − 1-Stage.

In [ ]:
# Tampilkan metrik 5: Token dan TEP
show_metric(4)

### 13.6 Query Variance Testing (QVT)

Kestabilan jawaban pada pasangan pertanyaan parafrase. Grup yang semua variasinya salah tidak dihitung.

In [ ]:
# Tampilkan metrik 6: QVT
show_metric(5)

### 13.7 Ringkasan dan Diagnostik SQL

Tabel pendukung untuk membaca keenam metrik. Tabel di sini memakai nama kode: Baseline = 1-Stage, GraphRAG = 2-Stage.

In [ ]:
# Bobot token output pada rumus T
ALPHA = cfg.token_output_weight

# Tampilkan ringkasan semua angka utama dalam satu tabel
show(rt.overall_comparison(baseline_raw_logs, graphrag_raw_logs, ALPHA))
# Tampilkan EX, ESM, dan token per tingkat kesulitan
show(rt.by_difficulty(baseline_raw_logs, graphrag_raw_logs, ALPHA))
# Tampilkan pengecekan prediksi: fallback, kosong, dan identik dengan gold
show(rt.prediction_checks(baseline_raw_logs, graphrag_raw_logs))
# Tampilkan jumlah query per kombinasi EX dan ESM
show(rt.outcome_matrix(baseline_raw_logs, graphrag_raw_logs))
# Tampilkan perbandingan EX per query antar varian
show(rt.paired_ex(baseline_raw_logs, graphrag_raw_logs))
# Tampilkan pemakaian fitur SQL pada prediksi
show(rt.sql_feature_usage(baseline_raw_logs, graphrag_raw_logs))

## 14. Cross-check dengan Evaluator Resmi Spider (subprocess)

Sebagai pembanding independen, panggil `external/spider_eval/evaluation.py` (resmi, tidak diubah) pada file prediksi. Satu hal penting: **parser resmi Spider hanya mengenali `JOIN` polos**, jadi `INNER/LEFT/RIGHT/FULL/CROSS JOIN` membuatnya crash. Karena itu file dinormalisasi dulu lewat `normalize_sql_file_for_parsing()` (`IMPLEMENTATION_DECISIONS.md` poin 11):

- `--etype match`: normalisasi **penuh** (ESM tidak pernah mengeksekusi SQL)
- `--etype exec`: normalisasi **hanya INNER/CROSS JOIN** (`execution_safe_only=True`), karena SQL benar-benar dieksekusi dan `LEFT/RIGHT/FULL JOIN` tidak boleh diam-diam berubah makna

Konsekuensinya, jalur subprocess ini tetap gagal parse untuk outer JOIN (**known limitation**); angka in-process di Bagian 13 tidak punya batasan itu. Angka Bagian 13 adalah sumber utama; bagian ini adalah cross-check. Hasil parse disimpan ke `spider_official_eval.csv` (dibaca export Excel).

In [ ]:
# Fungsi untuk menjalankan evaluator resmi Spider pada satu file prediksi
def run_spider_official(pred_path: Path, label: str) -> pd.DataFrame:
    """Jalankan evaluation.py resmi (match + exec) dan kembalikan skornya sebagai tabel."""
    # Folder file sementara
    tmp = ARTIFACT_DIR / "spider_eval_tmp"
    # Buat foldernya kalau belum ada
    tmp.mkdir(parents=True, exist_ok=True)
    # Ambil nama file prediksi tanpa ekstensi
    stem = pred_path.stem

    # Siapkan file gold dan prediksi untuk tiap jenis evaluasi
    inputs = {
        # Evaluasi struktur (ESM): semua jenis JOIN boleh disamakan
        "match": (
            # File gold yang sudah dinormalisasi
            normalize_sql_file_for_parsing(gold_path, tmp / f"{stem}_match_gold.txt", has_db_id_suffix=True,  execution_safe_only=False),
            # File prediksi yang sudah dinormalisasi
            normalize_sql_file_for_parsing(pred_path, tmp / f"{stem}_match_pred.txt", has_db_id_suffix=False, execution_safe_only=False),
        ),
        # Evaluasi eksekusi (EX): hanya INNER dan CROSS JOIN yang disamakan
        "exec": (
            # File gold yang sudah dinormalisasi
            normalize_sql_file_for_parsing(gold_path, tmp / f"{stem}_exec_gold.txt", has_db_id_suffix=True,  execution_safe_only=True),
            # File prediksi yang sudah dinormalisasi
            normalize_sql_file_for_parsing(pred_path, tmp / f"{stem}_exec_pred.txt", has_db_id_suffix=False, execution_safe_only=True),
        ),
    }
    # Tempat hasil tiap jenis evaluasi
    parts = []
    # Jalankan kedua jenis evaluasi
    for etype, (g_file, p_file) in inputs.items():
        # Panggil evaluation.py sebagai program terpisah
        r = subprocess.run(
            # Program yang dijalankan
            [sys.executable, "external/spider_eval/evaluation.py",
             # File gold dan prediksi
             "--gold", str(g_file), "--pred", str(p_file),
             # Folder database, schema, dan jenis evaluasi
             "--db", str(cfg.db_dir), "--table", str(cfg.tables_json), "--etype", etype],
            # Tangkap outputnya sebagai teks
            capture_output=True, text=True,
        )
        # Ubah output teks menjadi tabel
        parsed = rt.parse_spider_output(r.stdout)
        # Beri peringatan kalau evaluator gagal atau skornya tidak terbaca
        if r.returncode != 0 or parsed.empty:
            # Tampilkan varian, jenis evaluasi, dan kode keluar
            print(f"[WARN] {label} --etype {etype}: exit code {r.returncode}, {len(parsed)} baris skor terbaca")
            # Tampilkan akhir output untuk membantu mencari penyebabnya
            print(r.stdout[-1500:] or r.stderr[-1500:])
        # Simpan tabel skor
        parts.append(parsed)
    # Gabungkan skor ESM dan EX
    return pd.concat(parts, ignore_index=True)

# Jalankan evaluator resmi untuk kedua varian
spider_parsed = {"GraphRAG": run_spider_official(g_pred_path, "GraphRAG"),
                 # Varian 1-Stage
                 "Baseline": run_spider_official(b_pred_path, "Baseline")}
# Tampilkan ringkasan skor resmi
show(rt.spider_summary(spider_parsed))
# Tambahkan nama varian pada tiap tabel yang berisi skor
spider_frames = [df.assign(Pipeline=label) for label, df in spider_parsed.items() if not df.empty]
# Lanjutkan kalau ada skor yang terbaca
if spider_frames:
    # Gabungkan semua skor menjadi satu tabel
    spider_long = pd.concat(spider_frames, ignore_index=True)
    # Pindahkan kolom Pipeline ke paling depan
    spider_long = spider_long[["Pipeline"] + [c for c in spider_long.columns if c != "Pipeline"]]
    # Simpan skor resmi ke CSV
    spider_long.to_csv(ARTIFACT_DIR / "tables" / "spider_official_eval.csv", index=False)
    # Ambil skor partial match F1
    partial_f1 = spider_long[spider_long["Section"] == "Partial match F1"]
    # Tampilkan skor partial match F1
    show(rt.with_formats(partial_f1.drop(columns="Section").reset_index(drop=True),
                         # Format angkanya tiga desimal
                         {c: "num3" for c in partial_f1.columns if c not in ("Pipeline", "Section", "Metric")}),
         # Judul tabel
         "Official Spider evaluation: partial match F1")

## 15. Analisis 6 Dimensi (integrasi metrics -> dimensions)

Bagian 13 menghasilkan keenam metrik. **Dimensi** adalah level di atasnya: menggabungkan beberapa metrik untuk menjawab satu pertanyaan penelitian. Semuanya dijalankan lewat satu fungsi, `run_dimension_analysis()` dari `pipeline.py` — **fungsi yang sama** dipanggil otomatis di akhir `python src/experiments/pipeline.py --baseline`, jadi hasil notebook dan hasil script CLI tidak bisa berbeda.

| Dimensi | Pertanyaan yang dijawab | Metrik yang digabung |
|---|---|---|
| **1. Efisiensi** | Apakah GraphRAG lebih hemat token tanpa mengorbankan akurasi? (RM1, hasil utama skripsi) | SLA, EX/ESM, Token Consumption, TEP |
| **2. Struktur SQL** | Bagaimana EX dan ESM berubah bersama? | ESM x EX, per difficulty |
| **3. Komponen SQL** | Klausa mana yang paling terdampak pruning konteks? | CM per klausa (SELECT/WHERE/GROUP BY/...) |
| **4. Robustness** | Apakah jawaban tetap konsisten saat pertanyaan diparafrase? | QVT (+ silang ESM) |
| **5. Bottleneck** | Kalau akurasi rendah, gagal di retrieval (tabel/kolom) atau di generation? | SLA table-level & column-level x EX |
| **6. Ablation few-shot** | Berapa k few-shot yang cukup (elbow point)? | EX per k, Token Consumption per k |

**Aturan penting (keputusan di `IMPLEMENTATION_DECISIONS.md` poin 21, 23, 24):**

- **Dimensi 4 hanya jalan pada run penuh** (`SAMPLE_RATIO = 1.0`). Data QVT dibangun otomatis dari paraphrase alami `dev.json` (~470 SQL, masing-masing punya 2 pertanyaan), di-join ke `raw_logs` lewat `query_id` = posisi baris di `dev.json`. Pada run trial dimensi ini dilewati dengan sengaja supaya tidak tercampur data QVT lama.
- **Dimensi 5 bersyarat**: hanya dijalankan kalau Dimensi 2 menemukan kuadran "EX rendah, ESM rendah" (`escalate_to_dimension_5`). `ALWAYS_DIM5 = True` menimpa aturan ini.
- **Dimensi 6 butuh `EX_PER_K`** dari ablation tahap 2. `ablation.py` sekarang menghitung EX per k in-process dan menulis `outputs/tables/ablation_ex_per_k_graphrag_skeleton.json` (seleksi `skeleton`); sel di bawah membacanya otomatis kalau file itu ada (atau isi manual, mis. `{0: 45.2, 1: 52.1, 3: 54.0, 5: 53.8}`, skala 0-100). Kalau tidak ada, Dimensi 6 di-skip dengan penjelasan. Ablation cukup dijalankan dengan satu model, jadi angka Dimensi 6 sama di ketiga model.
- Dimensi yang prasyaratnya belum ada **di-skip dengan alasan**, bukan crash.
- Laporan teks dimensi tidak dicetak di sini (`verbose=False`); angka-angkanya ditampilkan sebagai tabel di 15b. File `dimensions_results.json` dan `dimensions_report.txt` tetap ditulis seperti biasa.

In [ ]:
# Dimensi 6 memakai EX per k; None berarti dibaca dari hasil ablation.py
EX_PER_K    = None    # Dimensi 6: {k: EX 0-100}. None -> dibaca dari hasil ablation.py kalau ada
# ablation.py menamai file ini ablation_ex_per_k_graphrag_<seleksi>.json; hanya seleksi "question" yang tanpa akhiran
# Tentukan akhiran nama file sesuai cara seleksi few-shot
_tag = "graphrag" if cfg.few_shot_selection == "question" else f"graphrag_{cfg.few_shot_selection}"
# Lokasi file hasil ablation
EX_PER_K_FILE = Path(f"outputs/tables/ablation_ex_per_k_{_tag}.json")
# Baca filenya kalau EX_PER_K belum diisi dan filenya ada
if EX_PER_K is None and EX_PER_K_FILE.exists():
    # Ubah isi file menjadi {k: EX}
    EX_PER_K = {int(k): v for k, v in json.loads(EX_PER_K_FILE.read_text()).items()}
    # Tampilkan nilai yang terbaca
    print(f"EX_PER_K dari {EX_PER_K_FILE}: {EX_PER_K}")
# False berarti Dimensi 5 mengikuti hasil Dimensi 2
ALWAYS_DIM5 = False   # True -> paksa Dimensi 5 tanpa menunggu trigger dari Dimensi 2

# Menjalankan Dimensi 1-6 (data QVT sudah dibangun di Bagian 13),
# dan menulis dimensions_results.json + dimensions_report.txt ke ARTIFACT_DIR/tables/
dim_results = run_dimension_analysis(
    # Folder raw_logs model ini
    raw_logs_dir=RAW_LOGS_DIR,
    # Folder hasil dimensi
    out_dir=ARTIFACT_DIR / "tables",
    # File dev set
    dev_json=cfg.dev_json,
    # Dimensi 4 hanya jalan pada run penuh
    full_dev=IS_FULL,
    # Data Dimensi 6
    ex_per_k=EX_PER_K,
    # Aturan Dimensi 5
    always_dim5=ALWAYS_DIM5,
    # Folder data QVT model ini
    qvt_dir=QVT_DIR,
    # Jangan cetak laporan teks; angkanya ditampilkan sebagai tabel
    verbose=False,
    # Data QVT tidak perlu dibangun dua kali
    build_qvt=False,
)
# Tampilkan status tiap dimensi
show(rt.dim_status(dim_results))

### 15b. Tabel per Dimensi

Angka dari `dimensions_results.json`, satu kelompok tabel per dimensi yang berstatus `ok`. Hanya nilai terukur yang ditampilkan; field teks interpretasi di JSON tidak ditampilkan.

| Dimensi | Tabel |
|---|---|
| 1 | Token consumption (mean/median) dan EX per difficulty; ΔEX, ΔT, TEP |
| 2 | ESM dan EX per difficulty dengan selisihnya |
| 3 | CM per klausa |
| 4 | QVT per pipeline, ΔQVT, tabulasi silang QVT × ESM |
| 5 | SLA (tabel & kolom) dan EX terhadap ambang proposal (F1 ≥ 0.80, EX ≥ 0.65) |
| 6 | EX per k dan selisih terhadap k sebelumnya |

In [ ]:
# Proses Dimensi 1 sampai 6
for n, name in rt.DIM_NAMES.items():
    # Ubah hasil dimensi itu menjadi tabel
    tables = rt.DIM_TABLES[n](dim_results)
    # Kalau tidak ada tabel, dimensi itu dilewati
    if not tables:
        # Ambil catatan dimensi itu
        rec = dim_results.get(f"dimension_{n}") or {}
        # Tampilkan nama dan statusnya
        print(f"D{n} {name}: {rec.get('status', 'not run')}")
        # Lanjut ke dimensi berikutnya
        continue
    # Proses tiap tabel dimensi itu
    for t in tables:
        # Tampilkan tabelnya
        show(t)

### 15c. Menjalankan Ulang Hanya Lapisan Analisis

Kalau `raw_logs` sudah ada (mis. dari run Kaggle sebelumnya) dan yang berubah hanya konfigurasi analisis (mis. `EX_PER_K` baru dari ablation), **tidak perlu generate ulang**. Contoh untuk model 14B:

```bash
python src/experiments/run_all_dimensions.py --raw-logs-dir data/raw_logs/Qwen2.5-Coder-14B-Instruct --qvt-dir data/qvt_variations/Qwen2.5-Coder-14B-Instruct --out-dir outputs/models/Qwen2.5-Coder-14B-Instruct/tables --ex-per-k-file outputs/tables/ablation_ex_per_k_graphrag_skeleton.json
```

atau ulangi hanya sel 15 di atas.

### 15d. Simpan Semua Hasil

`src/experiments/export_excel.py` menulis **satu workbook per model** dari file hasil run, memakai tabel yang sama dengan notebook ini (`result_tables.py`). Urutan sheet mengikuti alur notebook: metrik, dimensi, lalu rincian pendukung.

| Sheet | Isi |
|---|---|
| Info | sumber data yang terbaca, parameter run, definisi metrik |
| M1–M6 | satu sheet per metrik: SLA, ESM, EX, CM, Token + TEP, QVT |
| Dimensions, D1-D6 | status dan tabel angka tiap dimensi |
| Overall, By Difficulty, Retrieval | ringkasan dan rincian pendukung |
| SQL Diagnostics | cek prediksi, EX × ESM, EX berpasangan, fitur SQL, CM per klausa, confound diagnostics |
| Spider Official | hasil `evaluation.py` resmi (Bagian 14) |
| Per Query | satu baris per query: SQL, EX/ESM, token, SLA kolom kedua varian |

Semua hasil model ini ada di `outputs/models/<model>/` (run percobaan: `outputs/trial/models/<model>/`). Di Kaggle, sel di bawah juga membuat `hasil_skripsi_<MODEL>.xlsx` dan `hasil_<MODEL>.zip` di `/kaggle/working/` untuk diunduh.

### 15e. Menggabungkan Ketiga Model

Setelah 3B, 7B, dan 14B selesai, kumpulkan hasil ekstrak zip dari tiap orang ke `outputs/models/`, lalu jalankan:

```bash
python src/experiments/combine_models.py
```

Hasilnya `outputs/tables/hasil_gabungan.xlsx` dan `hasil_gabungan.json`. Skrip ini memberi peringatan kalau ada model yang belum ada, run percobaan, atau konfigurasi yang berbeda antar model.

In [ ]:
# shutil untuk menyalin folder dan membuat zip
import shutil
# Dipakai untuk membaca kembali workbook yang baru dibuat
from openpyxl import load_workbook

# Lokasi file Excel model ini
XLSX = ARTIFACT_DIR / "tables" / "hasil_skripsi.xlsx"
# Jalankan export_excel.py untuk membuat workbook
subprocess.run([sys.executable, "src/experiments/export_excel.py", "--raw-logs-dir", str(RAW_LOGS_DIR),
                # Folder sumber, lokasi hasil, dan buang sheet yang datanya tidak ada
                "--tables-dir", str(ARTIFACT_DIR / "tables"), "--out", str(XLSX), "--skip-missing"], check=True)
# Buka workbook untuk diperiksa
_wb = load_workbook(XLSX)
# Tampilkan daftar sheet beserta ukurannya
show(pd.DataFrame([{"Sheet": ws.title, "Tables": len(ws.tables), "Rows": ws.max_row, "Columns": ws.max_column} for ws in _wb]),
     # Judul tabel
     f"Workbook: {XLSX}")

# Lokasi salinan raw_logs di dalam folder hasil
_raw_copy = ARTIFACT_DIR / "raw_logs"
# Pada run penuh, raw_logs ada di folder lain, jadi perlu disalin
if RAW_LOGS_DIR != _raw_copy:
    # Salin raw_logs supaya ikut masuk zip
    shutil.copytree(RAW_LOGS_DIR, _raw_copy, dirs_exist_ok=True)

# Bagian ini hanya berjalan di Kaggle
if Path("/kaggle/working").exists():
    # Salin Excel supaya mudah diunduh
    shutil.copy(XLSX, f"/kaggle/working/hasil_skripsi_{MODEL}.xlsx")
    # Buat zip berisi seluruh hasil model ini
    _zip = shutil.make_archive(f"/kaggle/working/hasil_{MODEL}", "zip", root_dir=ARTIFACT_DIR.parent, base_dir=ARTIFACT_DIR.name)
    # Tampilkan file yang bisa diunduh
    print(f"Unduh dari /kaggle/working: hasil_skripsi_{MODEL}.xlsx dan {Path(_zip).name}")

## Catatan Desain yang Perlu Disebut di Metodologi

- **Few-shot sama untuk kedua pipeline.** Baseline dan GraphRAG sama-sama memakai `cfg.few_shot_k` contoh dari index yang sama (`build_few_shot_block()`), jadi selisih GraphRAG–Baseline murni dari retrieval. k final dipilih di ablation tahap 2 (EX tertinggi, rata-rata kedua pipeline).
- **Seleksi GraphRAG berbasis threshold**: tabel dan kolom disaring dengan skor cosine minimum, tanpa batas top-k%; threshold dipilih dengan SLA di `graphrag_sla_trial.ipynb`. Langkah lain (nama natural, stopword `edge`, value linking teks, `fk_only`, sambung tabel tanpa PK) adalah **preprocessing tetap**, kontribusi masing-masing ditunjukkan rekap trial G1-G7 di notebook itu. Baseline memilih tabel dengan cara yang sama (n-gram, threshold tabel, tanpa top-k%), hanya tanpa tahap kolom: seluruh kolom tabel terpilih dikirim.
- **Traversal `fk_only`.** Dari jalur traversal hanya kolom JOIN yang masuk prompt; tabel tanpa PK disambung di graph traversal supaya JOIN lewat tabel penghubung terbaca.
- **Nama natural Spider** (`embed_natural_names`) dipakai sebagai teks embed GraphRAG **dan** Baseline (preprocessing data schema yang sama, jadi selisih hasil murni dari cara retrieval); anotasinya berasal dari `tables.json` Spider sendiri. SQL dan DDL tetap nama asli.
- **Few-shot skeleton** (`few_shot_selection="skeleton"`): contoh dipilih dengan kemiripan pertanyaan yang domain-nya di-mask (de-semanticization, Guo et al., 2023, disederhanakan); prompt tetap berisi pertanyaan + SQL asli contohnya. Sama untuk kedua pipeline.
- **Few-shot hanya dari `train_spider.json`.** Retrieval dari train set, bukan kebocoran dev/test.
- **1-Stage dan 2-Stage.** `baseline` di kode adalah GraphRAG 1-Stage, dan `graphrag` di kode adalah GraphRAG 2-Stage. Nama lama masih dipakai untuk file dan kunci data.
- **Proxy vs SLA.** Recall/Precision yang diprint pipeline adalah proxy name-matching, bukan SLA; SLA resmi dari `src/metrics/sla.py`.
- **ESM ≡ EM di Spider; EX order-sensitive** (`eval_exec_match()` resmi apa adanya). Diagnostik confound di Bagian 12 menghitung kasusnya tanpa mengubah metrik.